<a href="https://colab.research.google.com/github/evandwh/ST554---Spring-2026---NCSU/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2

Author: Evan Whitfield

Course: ST554 - Spring 2026

Purpose: Project 2

In [158]:
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan
from functools import reduce
from pyspark.sql.types import *
import pandas as pd

In [166]:
class SparkDataChecker:
    def __init__(self, dataframe):
        self.df = dataframe

    @classmethod
    def read_csv(cls, spark_session, file_path):
        df = spark.read.load(file_path, format="csv", header=True, inferSchema=True)
        return cls(df)

    @classmethod
    def from_pandas(cls, spark, pandas_df):
        df = spark.createDataFrame(pandas_df)
        return cls(df)

    def _is_numeric_column(self, col):
        numeric_types = ['int', 'bigint', 'double', 'float', 'long', 'decimal', 'longint', 'bigint']

        dtype_dict = dict(self.df.dtypes)

        if col not in dtype_dict:
            print(f"Column '{col}' does not exist.")
            return False

        return dtype_dict[col] in numeric_types

    def check_in_range(self, column_name, lower = None, upper = None):

        if lower is None and upper is None:
            print("Must give at least one of lower or upper.")
            return self

        if not self._is_numeric_column(column_name):
            print(f"Column '{column_name}' is not numeric.")
            return self

        if lower is None:
            lower = float("-inf")

        if upper is None:
            upper = float("inf")

        tf_list = self.df[column_name].between(lower, upper)

        # Handle NULL and NaN explicitly
        tf_list = when(col(column_name).isNull() | isnan(col(column_name)),
            None).otherwise(tf_list)

        self = self.df.withColumn(f"{column_name}_in_range", tf_list)

        return self


In [169]:
data = {'Name': ['Alice', 'Bob', 'Carl', None, 'Eve'], 'Age': [25, 30, None, 28, 22], 'City': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Miami']}
df = pd.DataFrame(data)

spark = SparkSession.builder.appName("Pandas to Spark").getOrCreate()

data2 = SparkDataChecker.from_pandas(spark, df)

data2.check_in_range('Age', lower = 29).show()

+-----+----+-----------+------------+
| Name| Age|       City|Age_in_range|
+-----+----+-----------+------------+
|Alice|25.0|   New York|       false|
|  Bob|30.0|Los Angeles|        true|
| Carl| NaN|    Chicago|        NULL|
| NULL|28.0|    Houston|       false|
|  Eve|22.0|      Miami|       false|
+-----+----+-----------+------------+

